# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from groq import Groq
import google.generativeai as genai

In [ ]:
#constants

Model_Groq = 'llama-3.3-70b-versatile'
Model_Gemini = 'gemini-3-flash'

In [3]:
# load API key(Groq)
load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and len(api_key)>10:
    print("API key looks good")

groq = Groq()

API key looks good


In [16]:
# Load API key(Gemini)
load_dotenv(override=True)
genai.configure(
    api_key=os.getenv("GOOGLE_API_KEY"),
    transport="rest"
)

model = genai.GenerativeModel(Model_Gemini)

In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [6]:
# Get Groq to answer
def ask_groq_tutor(question):
    response = groq.chat.completions.create(
        model = Model_Groq,
        messages = [
            {"role": "system", "content": "You are a technical tutor"},
            {"role": "user", "content": question}
        ],
        temperature = 0.7
    )

    return response.choices[0].message.content


In [7]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
print("GROQ Answer:\n")
groq_answer = ask_groq_tutor(question)
display(Markdown(groq_answer))


GROQ Answer:



**Explanation of the Code**

The given code is a generator expression in Python that yields a sequence of authors from a list of books. Here's a breakdown of what it does:

* `books` is assumed to be a list of dictionaries, where each dictionary represents a book.
* `book.get("author")` attempts to retrieve the value associated with the key `"author"` from each book dictionary. If the key is not present, it returns `None` by default.
* The dictionary comprehension `{book.get("author") for book in books if book.get("author")}` creates a set of unique authors. The `if book.get("author")` condition filters out books that do not have an author.
* `yield from` is a syntax in Python that allows a generator to yield values from another iterable. In this case, it yields each author in the set.

**Why this code is used**

This code is likely used to extract a list of unique authors from a collection of books. The reasons for using this approach are:

* **Efficiency**: Using a set to store authors eliminates duplicates, which can be beneficial when dealing with large datasets.
* **Conciseness**: The dictionary comprehension and `yield from` syntax provide a concise way to express the logic.
* **Memory efficiency**: By using a generator expression, the code avoids creating a large list of authors in memory. Instead, it yields each author one at a time, which can be beneficial when dealing with large datasets.

**Example Use Case**

Suppose you have a list of books, where each book is represented as a dictionary:
```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4"}  # no author
]
```
Using the given code, you can extract the unique authors like this:
```python
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for author in get_authors(books):
    print(author)
```
Output:
```
Author A
Author B
```
Note that the order of authors is not guaranteed, as sets are unordered in Python. If you need to preserve the order, you can use a different approach, such as using a list comprehension with a set to keep track of unique authors.

In [19]:
# Get Gemini to answer
def ask_gemini_tutor(question):
    response = model.generate_content(
        question,
        generation_config={
            "temperature" : 0.7
        }
    )

    return response.text

In [ ]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

print("GEMINI Answer:\n")
gemini_answer = ask_gemini_tutor(question)
display(Markdown(gemini_answer))

In [24]:
#Using ollama

import ollama

def ask_ollama_tutor(question):
    response = ollama.chat(
        model='phi3',
        messages=[
            {"role": "system", "content": "You are a technical tutor"},
            {"role": "user", "content": question}
        ]
    )
    return response['message']['content']

In [ ]:
!ollama run phi3
!ollama serve

In [25]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

print("OLLAMA Answer:\n")
ollama_answer = ask_ollama_tutor(question)
display(Markdown(ollama_answer))

OLLAMA Answer:



This line of Python uses the `yield` statement to create an iterable generator expression within a context where it might not be immediately obvious, such as inside another function or comprehension that doesn't directly use generators. Let me break down what this code is doing step by step:

1. The outermost structure seems like dictionary unpacking (`yield from {...}`), but in Python 3.3 and newer versions (though not exactly correct syntactically as `from` cannot be used with generators directly, it would typically look for a generator expression if this was valid code). In its intended formulation without syntax errors:
   ```python
   yield from ({book["author"] for book in books if book.get("author")})  # Intended to create an iterable of authors' names where the author is not None or empty, but generators and `yield` don't work this way directly without appropriate function wrappers/decorations like `@generator`.
   ```
2. The generator expression `{book["author"] for book in books if book.get("author")}` creates a new dictionary where each key is an author’s name from the "books" list, and its value will be `None` because dictionaries do not hold duplicate keys; this means it effectively enumerates distinct authors across all provided books without repetition of empty or non-existent 'author' data.
3. The condition inside the generator (`if book.get("author")`) ensures that only those items where there is a valid author entry (i.e., not `None` or an empty string which might imply missing information) are considered and yielded from this expression, although as mentioned in point 1 above with proper Python syntax it wouldn't directly create such dictionaries but would rather return each qualifying 'author'.
4. As of the knowledge cutoff date (2023), to correctly use `yield` within a function or comprehension and yield values from another generator, you typically need some sort of wrapper like this:
   ```python
   def author_generator(books):
       for book in books:
           if book.get("author"):
               yield book["author"]
   # Example usage (assuming the use of a decorator or function to treat it as an iterable generator)
   
   @itertools.accumulate  # This is not actually how you accumulate in Python, but for illustration purposes only:
   def gen_authors(books):
       return author_generator(books)
   
   authors = list(gen_authors(books))
   ```
5. The `yield from` construction was intended to extract all values yielded by another iterator or generator (e.g., the inner iterable). If this were syntactically and functionally correct, it would effectively flatten the results of the comprehension into a single stream of authors' names without duplicates based on non-falsy `author` entries in each book dictionary within 'books'.

In summary, if unmodified by syntax errors or additional contexts like `@generator`, this code snippet seems to intend for creating an iterable that yields author names from books where the name is valid (not empty) but without duplication due to the nature of dictionaries in Python. However, actual generator expression and `yield` usage would typically be encapsulated within a function or require proper decorators/contexts like `@generator`.

Please adjust this explanation according to your specific syntax rules if working with an older version of Python or different context not covered herein (e.g., using `.items()` instead of dictionary comprehension, `yield from` in actual generator functions).